# Recalculate measured results



In [1]:
from pathlib import Path
from utils.datasets import flat_result_path, flat_result_directory
import h5py
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
from utils.datasets import load_npz, save_npz
from utils.experimental_analysis import (
    _manifest, _indices, _load_features, _load_source, _find, _normalise,
    _ridge_output, _ridge_accuracy, _summary, _matlab_text, _scalar,
    measurement_summary, FMNIST_CLASSES,
)

ROOT = Path.cwd()
if not (ROOT / 'experimental_results').is_dir():
    raise FileNotFoundError('Open this notebook from the project root containing experimental_results/.')
RESULTS = ROOT / 'results/experimental'
flat_result_directory(RESULTS).mkdir(parents=True, exist_ok=True)


## 1. Fmnist


In [2]:
root = ROOT / 'experimental_results/fmnist/figure2'
ipc_values = np.array([1, 2, 5, 10, 20, 50, 100], dtype=np.int32)
reps = np.arange(1, 10, dtype=np.int32)
feature_files = ([root / f'rep{rep}.mat' for rep in reps]
                 + [root / f'rep{rep}_exp2.mat' for rep in reps])
source_files = [root / 'sim_source_data' / f'rep{rep}.mat' for rep in reps] * 2
source_reps = np.tile(reps, 2)
acquisition_ids = np.repeat([1, 2], 9).astype(np.int32)
shared_seed = 17
shared_channels = np.sort(np.random.default_rng(shared_seed).choice(2025, 1024, replace=False))
channel_indices = [shared_channels if i in (13, 16) else None for i in range(18)]


In [3]:
coreset, random, random_extra, simulation = [], [], [], []
coreset_ipc1_confusions, random_ipc1_confusions = [], []
original_output_dims, selected_output_dims = [], []

with threadpool_limits(limits=4):
    for feature_path, source_path, channels in zip(feature_files, source_files, channel_indices):
        with h5py.File(feature_path, 'r') as handle:
            original_output_dims.append(int(np.prod(handle['output'].shape[:2])))
        features = _load_features(feature_path, channels)
        selected_output_dims.append(features.shape[1])
        soft, hard, blocks, random_positions = _load_source(source_path)
        assert len(features) == len(hard)
        test_index = _indices(_find(blocks, block_type='test'))
        test = _normalise(features[test_index])
        truth = hard[test_index]
        random_block = _find(blocks, variant='random')
        random_start = int(random_block['start_idx']) - 1
        coreset_row, random_row, simulation_row = [], [], []

        for ipc in ipc_values:
            block = _find([row for row in blocks if int(row['ipc']) == int(ipc)], variant='x_plus_y')
            supports = [('coreset', _indices(block), float(block['ridge_beta'])),
                        ('random', random_start + random_positions[int(ipc)] - 1, 1.0)]
            for method, indices, beta in supports:
                support = _normalise(features[indices])
                scores = _ridge_output(support, soft[indices], test, beta)
                prediction = scores.argmax(axis=1)
                accuracy = np.mean(prediction == truth)
                (coreset_row if method == 'coreset' else random_row).append(accuracy)
                if ipc == 1:
                    counts = np.zeros((10, 10), dtype=np.int64)
                    np.add.at(counts, (truth, prediction), 1)
                    (coreset_ipc1_confusions if method == 'coreset' else random_ipc1_confusions).append(counts)
            simulation_row.append(float(block['sim_acc_full']))

        new600 = _indices(_find(blocks, variant='random_unique_new'))
        union700 = np.concatenate((_indices(random_block), new600))
        extra_scores = []
        for indices in [new600, union700]:
            scores = _ridge_output(_normalise(features[indices]), soft[indices], test, 1.0)
            extra_scores.append(np.mean(scores.argmax(axis=1) == truth))
        coreset.append(coreset_row)
        random.append(random_row)
        random_extra.append(extra_scores)
        simulation.append(simulation_row)
coreset, random, random_extra, simulation = map(np.asarray, (coreset, random, random_extra, simulation))


In [4]:
result = {
    "ipc_values": ipc_values,
    "extended_random_ipc_values": np.array([600, 700], dtype=np.int32),
    "random_curve_ipc_values": np.concatenate((ipc_values, np.array([600, 700], dtype=np.int32))),
    "source_files": np.asarray([path.name for path in feature_files]),
    "source_reps": source_reps,
    "acquisition_ids": acquisition_ids,
    "class_names": FMNIST_CLASSES,
    "test_size": np.asarray(len(truth), dtype=np.int32),
    "normalization": np.asarray("independent_global_max_per_train_and_test_block"),
    "beta_policy": np.asarray("source_beta_for_coreset; beta_1_for_random"),
    "original_output_dim": np.asarray(original_output_dims, dtype=np.int32),
    "selected_output_dim": np.asarray(selected_output_dims, dtype=np.int32),
    "simulation_x_plus_y_accuracy": simulation,
    "simulation_x_plus_y_mean": simulation.mean(axis=0),
    "simulation_x_plus_y_std": simulation.std(axis=0, ddof=1),
    "random_curve_accuracy": np.concatenate((random, random_extra), axis=1),
}
_summary(result, "x_plus_y", coreset)
_summary(result, "random", random)
_summary(result, "extended_random", random_extra)

best = int(np.argmax(coreset[:, 0]))
selected = int(np.flatnonzero(result['source_files'] == 'rep5_exp2.mat')[0])
result.update({
    "best_ipc1_index": np.asarray(best, dtype=np.int32),
    "best_ipc1_source_file": result["source_files"][best],
    "best_ipc1_x_plus_y_accuracy": coreset[best, 0],
    "selected_ipc1_index": np.asarray(selected, dtype=np.int32),
    "selected_ipc1_source_file": result["source_files"][selected],
    "selected_ipc1_source_rep": result["source_reps"][selected],
    "selected_ipc1_acquisition_id": result["acquisition_ids"][selected],
    "selected_ipc1_x_plus_y_accuracy": coreset[selected, 0],
    "selected_ipc1_random_accuracy": random[selected, 0],
})
for name, matrices in (
    ("x_plus_y", coreset_ipc1_confusions),
    ("random", random_ipc1_confusions),
):
    counts = np.asarray(matrices)[selected]
    result[f"selected_ipc1_{name}_counts"] = counts
    result[f"selected_ipc1_{name}_percentages"] = 100.0 * counts / counts.sum(axis=1, keepdims=True)
result.update({
    "dataset": np.asarray("Fashion-MNIST, 9 scattering states x 2 acquisitions"),
    "shared_channel_subsample_indices": shared_channels.astype(np.int32),
    "confusion_selection_policy": np.asarray(
        "Figure 2(e,f): paired IPC=1 results from rep5_exp2.mat"
    ),
})
experiment = result
assert experiment['x_plus_y_accuracy'].shape == (18, 7)
assert int(experiment['test_size']) == 10000
save_npz(RESULTS / 'fmnist_experiment.npz', experiment)
display(measurement_summary(ipc_values, coreset, random))


,Samples per class,Coreset accuracy (%),Random accuracy (%)
0,1,79.68 ± 0.62,48.73 ± 5.94
1,2,80.83 ± 0.64,55.81 ± 2.98
2,5,82.30 ± 0.59,65.57 ± 2.14
3,10,82.82 ± 0.47,71.10 ± 1.22
4,20,83.45 ± 0.40,74.73 ± 1.09
5,50,83.88 ± 0.63,78.64 ± 0.70
6,100,84.32 ± 0.34,80.55 ± 0.46


### Matched-accuracy sample efficiency

Figure 2f uses arithmetic means with min-max error bars across nine acquisition-2 measurements. Standard deviations in the summary table use ddof=1.


In [5]:
from utils.experimental_analysis import sample_efficiency_table

RECOMPUTE_MATCHING = False
matching_path = RESULTS / 'fmnist_sample_efficiency.npz'
if RECOMPUTE_MATCHING or not flat_result_path(matching_path).is_file():
    ipc_values = (1, 2, 5, 10, 20)
    random_seeds = (0, 1, 2)
    with threadpool_limits(limits=4):
        root = Path(root)
        selected = experiment['acquisition_ids']==2
        files = experiment['source_files'][selected].astype(str)
        reps = experiment['source_reps'][selected].astype(int)
        targets = experiment['x_plus_y_accuracy'][selected,:len(ipc_values)]
        shared = experiment['shared_channel_subsample_indices'].astype(int)
        matched = np.empty_like(targets, dtype=np.int32)
        accuracies = np.empty_like(targets)
        lower_bound = np.zeros_like(targets, dtype=bool)
        coarse_ipc = np.arange(25,701,25)
        for row,(filename,rep) in enumerate(zip(files,reps)):
            channels = shared if filename in {'rep5_exp2.mat','rep8_exp2.mat'} else None
            features = _load_features(root/filename,channels)
            soft, hard, blocks, _ = _load_source(root/'sim_source_data'/f'rep{rep}.mat')
            test_index = _indices(_find(blocks,block_type='test'))
            test, truth = _normalise(features[test_index]), hard[test_index]
            random_block = _find(blocks,variant='random')
            pool = np.concatenate([_indices(random_block),_indices(_find(blocks,variant='random_unique_new'))])
            permutations=[]
            for seed in random_seeds:
                rng=np.random.default_rng(int(seed))
                permutations.append({c:rng.permutation(pool[hard[pool]==c]) for c in range(10)})
            cache={}
            def evaluate(ipc):
                if int(ipc) not in cache:
                    values=[]
                    for permutation in permutations:
                        indices=np.concatenate([permutation[c][:ipc] for c in range(10)])
                        score,_=_ridge_accuracy(features[indices],soft[indices],test,truth,float(random_block['ridge_beta']))
                        values.append(score)
                    cache[int(ipc)]=float(np.mean(values))
                return cache[int(ipc)]
            coarse=np.maximum.accumulate([evaluate(int(n)) for n in coarse_ipc])
            for col,target in enumerate(targets[row]):
                crossing=np.flatnonzero(coarse>=target)
                if not len(crossing):
                    budget=700;lower_bound[row,col]=True
                else:
                    upper_index=int(crossing[0]);upper=int(coarse_ipc[upper_index])
                    lower=0 if upper_index==0 else int(coarse_ipc[upper_index-1])
                    fine=np.arange(max(5,lower+5),upper+1,5)
                    hits=np.flatnonzero(np.maximum.accumulate([evaluate(int(n)) for n in fine])>=target)
                    budget=int(fine[hits[0]]) if len(hits) else upper
                matched[row,col]=budget;accuracies[row,col]=evaluate(budget)
        efficiency = dict(source_files=files,source_reps=reps,acquisition_ids=np.full(len(files),2),
                        coreset_ipc=np.asarray(ipc_values),coreset_accuracy=targets,
                        matched_random_ipc=matched,matched_random_accuracy_mean=accuracies,
                        is_lower_bound=lower_bound,aggregate_is_lower_bound=lower_bound.any(axis=0),
                        random_seeds=np.asarray(random_seeds),candidate_step=np.asarray(5),
                        maximum_candidate_ipc=np.asarray(700),
                        error_bar_policy=np.asarray('minimum to maximum across nine states'),
                        ridge_beta_policy=np.asarray('fixed beta from the random-100 source block'))
else:
    efficiency = load_npz(matching_path)
matched = efficiency['matched_random_ipc']
core = efficiency['coreset_ipc']
efficiency.update(matched_random_ipc_mean=matched.mean(axis=0),
                  matched_random_ipc_std=matched.std(axis=0,ddof=1),
                  matched_random_ipc_min=matched.min(axis=0),
                  matched_random_ipc_max=matched.max(axis=0),
                  sample_efficiency_gain=(matched/core).mean(axis=0),
                  sample_efficiency_gain_std=(matched/core).std(axis=0,ddof=1))
save_npz(matching_path, efficiency)
display(sample_efficiency_table(efficiency).round(2))


,Coreset IPC,Matched random IPC (mean),Matched random IPC (s.d.),Sample-efficiency gain (mean),Sample-efficiency gain (s.d.)
0,1,86.11,15.96,86.11,15.96
1,2,124.44,26.15,62.22,13.08
2,5,322.22,103.35,64.44,20.67
3,10,419.44,140.63,41.94,14.06


## 2. Ag news


In [6]:
dataset = 'agnews'
size_field, coreset_method = 'texts_per_class', 'x_plus_y'
confusion_size, confusion_rep = 5, 2
representative_acquisition = 'acquisition1'
random_uses_soft = False

recordings = {}
with threadpool_limits(limits=4):
    for acquisition in ['acquisition1', 'acquisition2']:
        root = ROOT / 'experimental_results' / dataset
        features = _load_features(root / f'feature_{acquisition}.mat')
        with h5py.File(root / 'sim_source_data/source.mat', 'r') as handle:
            soft = np.asarray(handle['all_soft_labels'], dtype=np.float64).T
            hard = np.asarray(handle['all_hard_labels'], dtype=np.int64).reshape(-1, order='F')
            blocks = _manifest(handle)
            names = np.asarray(handle['class_names'])
            class_names = np.array([''.join(chr(int(v)) for v in names[:, i]).rstrip()
                                    for i in range(names.shape[1])])
        assert len(features) == len(soft) == len(hard)
        test_blocks = [b for b in blocks if str(b['method']) == 'test']
        assert len(test_blocks) == 1
        test_index = _indices(test_blocks[0])
        test_raw, truth = features[test_index], hard[test_index]
        supports = [b for b in blocks if str(b['method']) != 'test']
        sizes = np.array(sorted({int(b[size_field]) for b in supports}), dtype=np.int32)
        reps = np.array(sorted({int(b['rep_index']) for b in supports}), dtype=np.int32)
        result = dict(dataset=np.asarray(dataset), size_name=np.asarray(size_field),
                      size_values=sizes, source_reps=reps, test_size=np.asarray(len(truth), dtype=np.int32),
                      normalization=np.asarray('shared_global_max_per_support_and_test_block'),
                      beta_policy=np.asarray('fixed_simulation_validation_beta'), class_names=class_names,
                      selected_confusion_size=np.asarray(confusion_size, dtype=np.int32),
                      selected_confusion_rep=np.asarray(confusion_rep, dtype=np.int32))
        for key, method in [('coreset', coreset_method), ('random', 'random')]:
            measured = np.empty((len(reps), len(sizes)))
            simulated = np.empty_like(measured)
            for row, rep in enumerate(reps):
                for column, size in enumerate(sizes):
                    matches = [b for b in supports if str(b['method']) == method
                               and int(b[size_field]) == size and int(b['rep_index']) == rep]
                    assert len(matches) == 1
                    block = matches[0]
                    index = _indices(block)
                    support_raw = features[index]
                    scale = max(float(support_raw.max()), float(test_raw.max()))
                    targets = soft[index]
                    if method == 'random' and not random_uses_soft:
                        targets = np.eye(len(class_names))[hard[index]]
                    scores = _ridge_output(support_raw / scale, targets, test_raw / scale, block['beta'])
                    prediction = scores.argmax(axis=1)
                    measured[row, column] = np.mean(prediction == truth)
                    simulated[row, column] = float(block['simulation_accuracy'])
                    if size == confusion_size and rep == confusion_rep:
                        counts = np.zeros((len(class_names), len(class_names)), dtype=np.int64)
                        np.add.at(counts, (truth, prediction), 1)
                        result[f'selected_{key}_accuracy'] = np.asarray(measured[row, column])
                        result[f'selected_{key}_confusion_counts'] = counts
                        result[f'selected_{key}_confusion_percentages'] = 100 * counts / counts.sum(axis=1, keepdims=True)
            _summary(result, key, measured)
            result[f'{key}_simulation_accuracy'] = simulated
        recordings[acquisition] = result


In [7]:
result = dict(recordings[representative_acquisition])
first = recordings['acquisition1']
for run in recordings.values():
    assert len(run['source_reps']) == 5
result['source_reps'] = np.concatenate([r['source_reps'] for r in recordings.values()])
result['source_acquisitions'] = np.concatenate([
    np.full(len(r['source_reps']), name) for name, r in recordings.items()])
result['selected_confusion_acquisition'] = np.asarray(representative_acquisition)
result['measured_repetitions'] = np.asarray(10, dtype=np.int32)
for method in ['coreset', 'random']:
    values = np.concatenate([r[method + '_accuracy'] for r in recordings.values()])
    _summary(result, method, values)
save_npz(RESULTS / f'{dataset}_experiment.npz', result)
display(measurement_summary(result['size_values'], result['coreset_accuracy'], result['random_accuracy']))

agnews = result


,Samples per class,Coreset accuracy (%),Random accuracy (%)
0,5,76.86 ± 1.24,39.96 ± 1.37
1,10,78.34 ± 0.76,46.64 ± 1.94
2,20,80.23 ± 0.76,54.61 ± 2.91
3,30,81.04 ± 0.60,60.81 ± 2.65
4,60,82.42 ± 0.37,71.35 ± 0.79
5,120,83.33 ± 0.34,77.77 ± 1.06


## 3. Cora


In [8]:
dataset = 'cora'
size_field, coreset_method = 'ipc', 'x_plus_y'
confusion_size, confusion_rep = 2, 2
representative_acquisition = 'acquisition2'
random_uses_soft = True

recordings = {}
with threadpool_limits(limits=4):
    for acquisition in ['acquisition1', 'acquisition2']:
        root = ROOT / 'experimental_results' / dataset
        features = _load_features(root / f'feature_{acquisition}.mat')
        with h5py.File(root / 'sim_source_data/source.mat', 'r') as handle:
            soft = np.asarray(handle['all_soft_labels'], dtype=np.float64).T
            hard = np.asarray(handle['all_hard_labels'], dtype=np.int64).reshape(-1, order='F')
            blocks = _manifest(handle)
            names = np.asarray(handle['class_names'])
            class_names = np.array([''.join(chr(int(v)) for v in names[:, i]).rstrip()
                                    for i in range(names.shape[1])])
        assert len(features) == len(soft) == len(hard)
        test_blocks = [b for b in blocks if str(b['method']) == 'test']
        assert len(test_blocks) == 1
        test_index = _indices(test_blocks[0])
        test_raw, truth = features[test_index], hard[test_index]
        supports = [b for b in blocks if str(b['method']) != 'test']
        sizes = np.array(sorted({int(b[size_field]) for b in supports}), dtype=np.int32)
        reps = np.array(sorted({int(b['rep_index']) for b in supports}), dtype=np.int32)
        result = dict(dataset=np.asarray(dataset), size_name=np.asarray(size_field),
                      size_values=sizes, source_reps=reps, test_size=np.asarray(len(truth), dtype=np.int32),
                      normalization=np.asarray('shared_global_max_per_support_and_test_block'),
                      beta_policy=np.asarray('fixed_simulation_validation_beta'), class_names=class_names,
                      selected_confusion_size=np.asarray(confusion_size, dtype=np.int32),
                      selected_confusion_rep=np.asarray(confusion_rep, dtype=np.int32))
        for key, method in [('coreset', coreset_method), ('random', 'random')]:
            measured = np.empty((len(reps), len(sizes)))
            simulated = np.empty_like(measured)
            for row, rep in enumerate(reps):
                for column, size in enumerate(sizes):
                    matches = [b for b in supports if str(b['method']) == method
                               and int(b[size_field]) == size and int(b['rep_index']) == rep]
                    assert len(matches) == 1
                    block = matches[0]
                    index = _indices(block)
                    support_raw = features[index]
                    scale = max(float(support_raw.max()), float(test_raw.max()))
                    targets = soft[index]
                    if method == 'random' and not random_uses_soft:
                        targets = np.eye(len(class_names))[hard[index]]
                    scores = _ridge_output(support_raw / scale, targets, test_raw / scale, block['beta'])
                    prediction = scores.argmax(axis=1)
                    measured[row, column] = np.mean(prediction == truth)
                    simulated[row, column] = float(block['simulation_accuracy'])
                    if size == confusion_size and rep == confusion_rep:
                        counts = np.zeros((len(class_names), len(class_names)), dtype=np.int64)
                        np.add.at(counts, (truth, prediction), 1)
                        result[f'selected_{key}_accuracy'] = np.asarray(measured[row, column])
                        result[f'selected_{key}_confusion_counts'] = counts
                        result[f'selected_{key}_confusion_percentages'] = 100 * counts / counts.sum(axis=1, keepdims=True)
            _summary(result, key, measured)
            result[f'{key}_simulation_accuracy'] = simulated
        recordings[acquisition] = result


In [9]:
result = dict(recordings[representative_acquisition])
first = recordings['acquisition1']
for run in recordings.values():
    assert len(run['source_reps']) == 5
result['source_reps'] = np.concatenate([r['source_reps'] for r in recordings.values()])
result['source_acquisitions'] = np.concatenate([
    np.full(len(r['source_reps']), name) for name, r in recordings.items()])
result['selected_confusion_acquisition'] = np.asarray(representative_acquisition)
result['measured_repetitions'] = np.asarray(10, dtype=np.int32)
for method in ['coreset', 'random']:
    values = np.concatenate([r[method + '_accuracy'] for r in recordings.values()])
    _summary(result, method, values)
save_npz(RESULTS / f'{dataset}_experiment.npz', result)
display(measurement_summary(result['size_values'], result['coreset_accuracy'], result['random_accuracy']))

cora = result


,Samples per class,Coreset accuracy (%),Random accuracy (%)
0,1,78.14 ± 1.29,34.54 ± 3.78
1,2,83.30 ± 1.74,44.56 ± 3.48
2,5,84.48 ± 0.98,57.90 ± 4.08
3,10,85.68 ± 1.04,69.98 ± 2.46
4,20,86.55 ± 0.77,75.30 ± 2.36


## 4. UCI-HAR


In [10]:
dataset = 'uci_har'
size_field, coreset_method = 'samples_per_class', 'distilled'
confusion_size, confusion_rep = 5, 4
representative_acquisition = 'acquisition1'
random_uses_soft = False

recordings = {}
with threadpool_limits(limits=4):
    for acquisition in ['acquisition1', 'acquisition2']:
        root = ROOT / 'experimental_results' / dataset
        features = _load_features(root / f'feature_{acquisition}.mat')
        with h5py.File(root / 'sim_source_data/source.mat', 'r') as handle:
            soft = np.asarray(handle['all_soft_labels'], dtype=np.float64).T
            hard = np.asarray(handle['all_hard_labels'], dtype=np.int64).reshape(-1, order='F')
            blocks = _manifest(handle)
            names = np.asarray(handle['class_names'])
            class_names = np.array([''.join(chr(int(v)) for v in names[:, i]).rstrip()
                                    for i in range(names.shape[1])])
        assert len(features) == len(soft) == len(hard)
        test_blocks = [b for b in blocks if str(b['method']) == 'test']
        assert len(test_blocks) == 1
        test_index = _indices(test_blocks[0])
        test_raw, truth = features[test_index], hard[test_index]
        supports = [b for b in blocks if str(b['method']) != 'test']
        sizes = np.array(sorted({int(b[size_field]) for b in supports}), dtype=np.int32)
        reps = np.array(sorted({int(b['rep_index']) for b in supports}), dtype=np.int32)
        result = dict(dataset=np.asarray(dataset), size_name=np.asarray(size_field),
                      size_values=sizes, source_reps=reps, test_size=np.asarray(len(truth), dtype=np.int32),
                      normalization=np.asarray('shared_global_max_per_support_and_test_block'),
                      beta_policy=np.asarray('fixed_simulation_validation_beta'), class_names=class_names,
                      selected_confusion_size=np.asarray(confusion_size, dtype=np.int32),
                      selected_confusion_rep=np.asarray(confusion_rep, dtype=np.int32))
        for key, method in [('coreset', coreset_method), ('random', 'random')]:
            measured = np.empty((len(reps), len(sizes)))
            simulated = np.empty_like(measured)
            for row, rep in enumerate(reps):
                for column, size in enumerate(sizes):
                    matches = [b for b in supports if str(b['method']) == method
                               and int(b[size_field]) == size and int(b['rep_index']) == rep]
                    assert len(matches) == 1
                    block = matches[0]
                    index = _indices(block)
                    support_raw = features[index]
                    scale = max(float(support_raw.max()), float(test_raw.max()))
                    targets = soft[index]
                    if method == 'random' and not random_uses_soft:
                        targets = np.eye(len(class_names))[hard[index]]
                    scores = _ridge_output(support_raw / scale, targets, test_raw / scale, block['beta'])
                    prediction = scores.argmax(axis=1)
                    measured[row, column] = np.mean(prediction == truth)
                    simulated[row, column] = float(block['simulation_accuracy'])
                    if size == confusion_size and rep == confusion_rep:
                        counts = np.zeros((len(class_names), len(class_names)), dtype=np.int64)
                        np.add.at(counts, (truth, prediction), 1)
                        result[f'selected_{key}_accuracy'] = np.asarray(measured[row, column])
                        result[f'selected_{key}_confusion_counts'] = counts
                        result[f'selected_{key}_confusion_percentages'] = 100 * counts / counts.sum(axis=1, keepdims=True)
            _summary(result, key, measured)
            result[f'{key}_simulation_accuracy'] = simulated
        recordings[acquisition] = result


In [11]:
result = dict(recordings[representative_acquisition])
first = recordings['acquisition1']
for run in recordings.values():
    assert len(run['source_reps']) == 5
result['source_reps'] = np.concatenate([r['source_reps'] for r in recordings.values()])
result['source_acquisitions'] = np.concatenate([
    np.full(len(r['source_reps']), name) for name, r in recordings.items()])
result['selected_confusion_acquisition'] = np.asarray(representative_acquisition)
result['measured_repetitions'] = np.asarray(10, dtype=np.int32)
for method in ['coreset', 'random']:
    values = np.concatenate([r[method + '_accuracy'] for r in recordings.values()])
    _summary(result, method, values)
save_npz(RESULTS / f'{dataset}_experiment.npz', result)
display(measurement_summary(result['size_values'], result['coreset_accuracy'], result['random_accuracy']))

uci_har = result


,Samples per class,Coreset accuracy (%),Random accuracy (%)
0,1,66.96 ± 2.28,43.57 ± 5.42
1,2,69.98 ± 1.10,47.26 ± 1.58
2,5,72.88 ± 1.75,56.05 ± 2.40
3,10,74.76 ± 0.74,60.57 ± 0.87
4,20,76.36 ± 0.92,62.99 ± 1.81
5,50,77.69 ± 0.96,67.08 ± 1.05


## 5. CCPP


In [12]:
ccpp_runs = []
selected = {}
with threadpool_limits(limits=4):
    for acquisition in ['acquisition1', 'acquisition2']:
        root = ROOT / 'experimental_results' / 'ccpp'
        features = _load_features(root / f'feature_{acquisition}.mat')
        with h5py.File(root / 'sim_source_data/source.mat', 'r') as handle:
            targets = np.asarray(handle['all_targets_standardized'], dtype=np.float64).reshape(-1, order='F')
            raw_targets = np.asarray(handle['all_test_targets_raw'], dtype=np.float64).reshape(-1, order='F')
            means = np.asarray(handle['y_mean_per_rep'], dtype=np.float64).reshape(-1, order='F')
            stds = np.asarray(handle['y_std_per_rep'], dtype=np.float64).reshape(-1, order='F')
            blocks = _manifest(handle)
        supports = [b for b in blocks if str(b['method']) != 'test']
        tests = [b for b in blocks if str(b['method']) == 'test']
        sizes = np.array(sorted({int(b['support_size']) for b in supports}), dtype=np.int32)
        reps = sorted({int(b['rep_index']) for b in supports})
        result = {'size_values': sizes}
        for key, method in [('coreset', 'distilled'), ('random', 'random')]:
            scores = np.empty((len(reps), len(sizes)))
            for row, rep in enumerate(reps):
                test_blocks = [b for b in tests if int(b['rep_index']) == rep]
                assert len(test_blocks) == 1
                test_index = _indices(test_blocks[0])
                truth = raw_targets[test_index]
                assert len(truth) == 1906
                for column, size in enumerate(sizes):
                    matches = [b for b in supports if str(b['method']) == method
                               and int(b['rep_index']) == rep and int(b['support_size']) == size]
                    assert len(matches) == 1
                    block = matches[0]
                    index = _indices(block)
                    scale = max(float(features[index].max()), 1e-12)
                    prediction = _ridge_output(features[index] / scale, targets[index, None],
                                               features[test_index] / scale, block['beta']).reshape(-1)
                    prediction = prediction * stds[row] + means[row]
                    scores[row, column] = 1 - np.sum((prediction - truth)**2) / np.sum((truth - truth.mean())**2)
                    if acquisition == 'acquisition2' and rep == 5 and size == 2:
                        selected['truth'] = truth
                        selected[key] = prediction
            result[f'{key}_r2'] = scores
        ccpp_runs.append(result)


In [13]:
ccpp = {'support_sizes': ccpp_runs[0]['size_values']}
for method in ['coreset', 'random']:
    ccpp[f'{method}_r2'] = np.concatenate([r[f'{method}_r2'] for r in ccpp_runs])
ccpp.update({f'selected_{key}': value for key, value in selected.items()})
save_npz(RESULTS / 'ccpp_experiment.npz', ccpp)
display(measurement_summary(ccpp['support_sizes'], ccpp['coreset_r2'], ccpp['random_r2'], metric='r2'))


,Training samples,Coreset R²,Random R²
0,2,0.8968 ± 0.0071,0.2839 ± 0.2425
1,4,0.9011 ± 0.0049,0.4028 ± 0.4534
2,8,0.9158 ± 0.0037,0.7634 ± 0.0617
3,12,0.9314 ± 0.0024,0.8161 ± 0.0517
4,16,0.9324 ± 0.0027,0.8415 ± 0.0317
5,24,0.9347 ± 0.0035,0.8715 ± 0.0301
6,32,0.9357 ± 0.0057,0.8982 ± 0.0195


## 6. NARMA10


In [14]:
from utils.regression_sample_efficiency import regression_inputs
narma_inputs = regression_inputs(ROOT, 'narma10')
target_mean = float(narma_inputs['y_mean'].item())
target_std = float(narma_inputs['y_std'].item())

narma_runs, selected = [], {}
with threadpool_limits(limits=4):
    for acquisition in ['acquisition1', 'acquisition2']:
        root = ROOT / 'experimental_results' / 'narma10'
        with h5py.File(root / 'sim_source_data/source.mat', 'r') as handle:
            truth = np.asarray(handle['origin_truth'], dtype=np.float64).T
            sizes = np.asarray(handle['support_sizes'], dtype=np.int32).reshape(-1, order='F')
        files = sorted((root / 'model_blocks').glob(f'model_*_{acquisition}.mat'))
        assert len(files) == 81
        records = []
        for model_id, path in enumerate(files, 1):
            with h5py.File(path, 'r') as handle:
                assert int(_scalar(handle, 'model_id')) == model_id
                assert _scalar(handle, 'block_completed')
                method = _matlab_text(handle['block/method'])
                origins = int(_scalar(handle, 'origin_count'))
                rep = int(_scalar(handle, 'block/rep_index'))
                size = int(_scalar(handle, 'block/support_size'))
                beta = float(_scalar(handle, 'beta'))

                support = np.asarray(handle['output_support'], dtype=np.float64).transpose(2, 1, 0).reshape(-1, 1024)
                camera = np.asarray(handle['output_autonomous'], dtype=np.float64).transpose(3, 2, 1, 0)[:origins]
                camera = camera.reshape(camera.shape[0], camera.shape[1], 1024)
                assert camera.shape[1] == 200
                scale = max(float(support.max()), 1e-12)
                targets = np.asarray(handle['support_targets_scaled'], dtype=np.float64).T
                prediction = _ridge_output(support / scale, targets, camera.reshape(-1, 1024) / scale, beta)
                prediction = prediction.reshape(camera.shape[:2]) * target_std + target_mean


                metadata = dict(model_id=model_id, method=method, support_size=size,
                                rep_index=rep, draw_id=int(_scalar(handle, 'block/draw_id')), beta=beta,
                                simulation_r2=float(_scalar(handle, 'block/simulation_autonomous_r2')))
                for origin in range(origins):
                    residual = prediction[origin] - truth[origin]
                    records.append(metadata | dict(
                        origin_id=origin + 1,
                        r2=1 - np.sum(residual**2) / np.sum((truth[origin] - truth[origin].mean())**2),
                        rmse=np.sqrt(np.mean(residual**2)), mae=np.mean(np.abs(residual)),
                    ))
                if acquisition == 'acquisition1' and rep == 3 and size == 12 and method in ['distilled', 'random']:
                    selected['truth'] = truth[1]
                    selected['coreset' if method == 'distilled' else 'random'] = prediction[1]

        frame = pd.DataFrame(records)
        result = dict(dataset=np.asarray('NARMA10'), size_name=np.asarray('support_size'),
                      size_values=sizes, source_reps=np.arange(1, 6, dtype=np.int32),
                      origin_count=np.asarray(2, dtype=np.int32),
                      beta_policy=np.asarray('fixed_simulation_validation_beta'),
                      evaluation=np.asarray('200_step_autonomous_rollout_two_fixed_origins'))
        for key, method in [('coreset', 'distilled'), ('random', 'random')]:
            for metric in ['r2', 'rmse', 'mae', 'simulation_r2']:
                columns = []
                for size in sizes:
                    rows = frame[(frame.method == method) & (frame.support_size == int(size))]
                    rows = rows.sort_values(['rep_index', 'origin_id'])
                    assert len(rows) == 10
                    columns.append(rows[metric].to_numpy())
                result[f'{key}_{metric}'] = np.column_stack(columns)
        full = frame[frame.method == 'full']
        assert len(full) == 1
        for metric in ['r2', 'rmse', 'mae', 'simulation_r2']:
            result[f'full_{metric}'] = np.asarray(full.iloc[0][metric])
        for name in ['model_id', 'method', 'support_size', 'rep_index', 'draw_id', 'origin_id',
                     'beta', 'simulation_r2', 'r2', 'rmse', 'mae']:
            result[f'per_run_{name}'] = frame[name].to_numpy()
        narma_runs.append(result)


In [15]:
narma = {'support_sizes': narma_runs[0]['size_values']}
for method in ['coreset', 'random']:
    values = [r[f'{method}_r2'].reshape(5, 2, len(narma['support_sizes'])).mean(axis=1)
              for r in narma_runs]
    narma[f'{method}_r2'] = np.concatenate(values)
assert set(selected) == {'truth', 'coreset', 'random'}
narma.update({f'selected_{key}': value for key, value in selected.items()})
save_npz(RESULTS / 'narma10_experiment.npz', narma)
display(measurement_summary(narma['support_sizes'], narma['coreset_r2'], narma['random_r2'], metric='r2'))


,Training samples,Coreset R²,Random R²
0,4,0.6931 ± 0.1190,-0.7170 ± 0.9500
1,8,0.7210 ± 0.0835,0.2700 ± 0.3715
2,12,0.7593 ± 0.1265,0.1640 ± 0.5689
3,16,0.8118 ± 0.0672,0.3273 ± 0.2632
4,24,0.8567 ± 0.0345,0.5826 ± 0.1920
5,32,0.8412 ± 0.0696,0.5679 ± 0.2495
6,48,0.8734 ± 0.0447,0.7834 ± 0.0442
7,64,0.9111 ± 0.0208,0.7299 ± 0.0835
